# DuckDB × Parquet

DuckDB は CSV だけでなく **Parquet** も直接クエリできます。  
Parquet は分析用途で広く使われるファイル形式で、Snowflake の内部フォーマットとも相性抜群です。

## 目次
1. CSV と Parquet の違い（なぜ Parquet か）
2. CSV → Parquet 変換
3. ファイルサイズ比較
4. Parquet を直接クエリ（`read_parquet()`）
5. **列指向の恩恵**：特定カラムだけ読む
6. 複数 Parquet をまとめてクエリ（ワイルドカード）

---
## 1. CSV と Parquet の違い

| 観点 | CSV | Parquet |
|------|-----|--------|
| データ構造 | 行指向（1行ずつ保存） | **列指向**（カラムごとに保存） |
| 型情報 | なし（全部文字列として読む） | **スキーマ付き** |
| 圧縮 | なし | **自動圧縮**（通常 3〜10倍小さい） |
| 特定列の読み込み | 全列読まないといけない | **必要な列だけ**読める |
| 用途 | 人間が読む・小データ | **分析・大規模データ**向け |

列指向とは？→ `amount` 列だけ集計したいとき、CSVは全行を読む必要があるが、Parquetは `amount` 列のデータだけ読めば済む。

In [ ]:
import duckdb
import os

DATA_DIR = "../data"
PARQUET_DIR = "../data/parquet"

# Parquet 出力先ディレクトリを作成
os.makedirs(PARQUET_DIR, exist_ok=True)

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

---
## 2. CSV → Parquet 変換

DuckDB は `COPY ... TO '...parquet'` 1行で変換できます。  
別途 Python ライブラリ（pyarrow など）は不要です。

In [ ]:
# 3つのCSVをParquetに変換
for table in ["customers", "products", "sales"]:
    con.execute(f"""
        COPY (
            SELECT * FROM read_csv_auto('{DATA_DIR}/{table}.csv')
        )
        TO '{PARQUET_DIR}/{table}.parquet'
        (FORMAT PARQUET)
    """)
    print(f"✓ {table}.parquet を作成しました")

---
## 3. ファイルサイズ比較

同じデータで CSV と Parquet のサイズを比べます。

In [ ]:
print(f"{'ファイル':<25} {'サイズ(bytes)':>15} {'備考'}")
print("-" * 55)

for table in ["customers", "products", "sales"]:
    csv_path     = f"{DATA_DIR}/{table}.csv"
    parquet_path = f"{PARQUET_DIR}/{table}.parquet"

    csv_size     = os.path.getsize(csv_path)
    parquet_size = os.path.getsize(parquet_path)
    ratio        = csv_size / parquet_size

    print(f"{table+'.csv':<25} {csv_size:>15,}")
    print(f"{table+'.parquet':<25} {parquet_size:>15,}  ← CSV の {ratio:.1f} 分の 1")
    print()

---
## 4. Parquet を直接クエリ

`read_csv_auto()` の代わりに `read_parquet()` を使うだけです。  
型情報がスキーマに含まれているので、日付型なども自動で正しく読まれます。

In [ ]:
# Parquet を直接 SELECT
rows = con.execute(f"""
    SELECT *
    FROM read_parquet('{PARQUET_DIR}/sales.parquet')
    LIMIT 5
""").fetchall()

print("=== sales.parquet の先頭5件 ===")
for row in rows:
    print(row)

In [ ]:
# スキーマ確認（型情報がきちんと入っているか）
schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET_DIR}/sales.parquet')
""").fetchall()

print("=== sales.parquet のスキーマ ===")
for col in schema:
    print(f"  {col[0]:<20} {col[1]}")

---
## 5. 列指向の恩恵：特定カラムだけ読む

Parquet は**列指向**なので、SELECT するカラムを絞ると、  
不要な列のデータはディスクから読まれません（I/O削減）。

DuckDB の `EXPLAIN ANALYZE` でどの列が読まれたか確認できます。

In [ ]:
# amount 列だけを使った集計
# → Parquet は amount 列のデータしか読まない（CSVは全列読む）
rows = con.execute(f"""
    SELECT
        COUNT(*)    AS 件数,
        SUM(amount) AS 売上合計,
        MIN(amount) AS 最小,
        MAX(amount) AS 最大,
        AVG(amount) AS 平均
    FROM read_parquet('{PARQUET_DIR}/sales.parquet')
""").fetchall()

row = rows[0]
print(f"件数     : {row[0]:,}")
print(f"売上合計 : {row[1]:,.0f} 円")
print(f"最小     : {row[2]:,.0f} 円")
print(f"最大     : {row[3]:,.0f} 円")
print(f"平均     : {row[4]:,.0f} 円")

In [ ]:
# Parquet のメタデータ（行数・列数・圧縮情報）を確認する
meta = con.execute(f"""
    SELECT *
    FROM parquet_metadata('{PARQUET_DIR}/sales.parquet')
""").fetchall()

print("=== Parquet メタデータ（1行グループ分） ===")
# カラム名も表示
cols = [desc[0] for desc in con.execute(f"""
    SELECT * FROM parquet_metadata('{PARQUET_DIR}/sales.parquet')
""").description]

for col, val in zip(cols, meta[0]):
    print(f"  {col:<30} {val}")

---
## 6. 複数 Parquet をまとめてクエリ

実務では月別・日別にファイルが分割されることが多いです。  
DuckDB はワイルドカード `*.parquet` で**複数ファイルを1クエリ**で処理できます。

まず sales データを月別に分割してみます。

In [ ]:
# 月別Parquetディレクトリ
MONTHLY_DIR = "../data/parquet/monthly"
os.makedirs(MONTHLY_DIR, exist_ok=True)

# 月の一覧を取得
months = con.execute(f"""
    SELECT DISTINCT strftime(sale_date, '%Y-%m') AS ym
    FROM read_parquet('{PARQUET_DIR}/sales.parquet')
    ORDER BY ym
""").fetchall()

# 月ごとに別ファイルへ出力
for (ym,) in months:
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_parquet('{PARQUET_DIR}/sales.parquet')
            WHERE strftime(sale_date, '%Y-%m') = '{ym}'
        )
        TO '{MONTHLY_DIR}/sales_{ym}.parquet'
        (FORMAT PARQUET)
    """)
    print(f"✓ sales_{ym}.parquet")

In [ ]:
# ワイルドカードで全月をまとめてクエリ
rows = con.execute(f"""
    SELECT
        strftime(sale_date, '%Y-%m') AS 年月,
        COUNT(*)                     AS 件数,
        SUM(amount)                  AS 売上合計
    FROM read_parquet('{MONTHLY_DIR}/*.parquet')
    GROUP BY 年月
    ORDER BY 年月
""").fetchall()

print(f"{'年月':<10} {'件数':>6} {'売上合計':>12}")
print("-" * 32)
for row in rows:
    print(f"{row[0]:<10} {row[1]:>6} {row[2]:>12,.0f}")

print(f"\n→ {len(months)} ファイルを1クエリで集計しました")

---
## まとめ

| 機能 | 説明 |
|------|------|
| `COPY (...) TO 'file.parquet' (FORMAT PARQUET)` | CSV → Parquet 変換 |
| `read_parquet('file.parquet')` | Parquet を直接クエリ |
| `read_parquet('dir/*.parquet')` | 複数ファイルをまとめてクエリ |
| `parquet_metadata('file.parquet')` | メタデータ（行数・圧縮情報）確認 |
| `DESCRIBE SELECT * FROM read_parquet(...)` | スキーマ確認 |

**Parquet の利点（再確認）**
- 圧縮でファイルが小さくなる → **ストレージ節約**
- 列指向で必要な列だけ読む → **I/O削減・高速クエリ**
- 型情報付き → **スキーマ推論が正確**
- S3 に置いた Parquet も同じ構文でクエリ可能（次のステップで学習）